# Static exciton propagator in the diagonal transition basis

This notebook evaluates the bare retarded electron-hole bubble and the dressed exciton interaction for the polarized two-layer model. The main calculation is organized in the transition-channel basis that diagonalizes the bare bubble,

$$
\lambda\in\{++ ,--,+-,-+\},
$$

where $\lambda$ labels the full electron-hole transition that diagonalizes the bare bubble. The momentum $k$ is not part of the channel label; it is summed inside each diagonal bubble component. The transition energy is

$$
\Delta E^\lambda(k)=E^\lambda_c(k)-E^\lambda_v(k).
$$

In this basis

$$
(\Pi^R_{\rm diag})^{\lambda\lambda'}(\Omega)
=\delta_{\lambda\lambda'}\,\frac{1}{L}\sum_k
\frac{f(E^\lambda_v(k))-f(E^\lambda_c(k))}{\Omega+i\eta-\Delta E^\lambda(k)}.
$$

The physical layer-pair fields

$$|11\rangle,\ |22\rangle,\ |12\rangle,\ |21\rangle$$

are obtained by the projection matrix $M$. They are still useful because the attraction is diagonal there,

$$V_\Phi=-\mathrm{diag}(V_{\rm intra},V_{\rm intra},V_{\rm inter},V_{\rm inter}),$$

but in the diagonal transition basis the same interaction is

$$V_{\rm diag}=M^\dagger V_\Phi M.$$

The propagator used for the main spectra is therefore

$$D^R_{\rm diag}(\Omega;\theta)=\left[V_{\rm diag}^{-1}-\Pi^R_{\rm diag}(\Omega;\theta)\right]^{-1}.$$

The layer-pair representation is kept as a benchmark/projection check, not as the main computational basis.


In [ ]:
using LinearAlgebra
using Printf
using PyPlot

rc("font", family="serif")
rc("font", size=13)
rc("axes", labelsize=14, titlesize=14)
rc("legend", fontsize=10)

# Shared plot colors for analytic branches / poles (used in Figs 2-4).
branch_colors = ["#2563eb", "#16a34a", "#f97316", "#dc2626"]
pole_colors   = ["cyan", "#7dd3fc", "white", "#f9a8d4"]

## Model and parameters

The one-particle Hamiltonian is block diagonal in the valence/conduction sector,

$$H_0(k,\theta)=P_v\otimes H_v(k,\theta)+P_c\otimes H_c(k,\theta).$$

The internal $2\times2$ space is the layer/polarized degree of freedom. We choose the scalar shifts so that the upper valence edge is the zero of energy, while the ferromagnetic bare gap is $\Delta_0$ and the gap grows toward the antiferromagnetic configuration.

With

$$c_\theta=\cos(\theta/2),\qquad J_{v\theta}=J_{pd,v}c_\theta,\qquad J_{c\theta}=J_{pd,c}c_\theta,$$

we use

$$H_v(k,\theta)=-(\epsilon_v(k)+2\gamma_v+J_{v\theta})\sigma_0+J_{v\theta}\sigma_x,$$

and

$$H_c(k,\theta)=\left[\epsilon_c(k)+2\gamma_c+\Delta_0+(J_{pd,c}+J_{pd,v})(1-c_\theta)+J_{c\theta}\right]\sigma_0-J_{c\theta}\sigma_x.$$

Thus

$$E_g(0)=\Delta_0,$$

and

$$E_g(\pi)=\Delta_0+J_{pd,c}+J_{pd,v}.$$

The interaction parameters below are positive attraction magnitudes. The kernel entering the BSE is negative.

In [ ]:
# -----------------------------
# Physical and numerical parameters  (units: eV)
# Calibrated to the A (bright, intralayer) exciton of CrSBr:
#   Diederich et al., "Magnon-mediated exciton-exciton interaction", arXiv:2409.18501
#   E_X(AFM) = 1.36 eV, red-shift AFM->FM = 15 meV, FM gap ~2.0 eV, AFM gap ~2.1 eV.
# Band/coupling ratios from the ab-initio four-band model:
#   Heissenbuettel et al., PRB 111, 075107 (2025) / arXiv:2403.20174.
# -----------------------------
L = 200                         # momentum points
γ_c = 0.40                      # conduction hopping; A = 2(γc+γv) = 1.6 eV continuum scale,
γ_v = 0.40                      # chosen so the fitted V_intra/V_inter matches ab-initio 2.48
Δ_0 = 2.0                       # ferromagnetic bare gap at k=0 (FM gap)

# Interlayer exchange/polarization couplings. NOTE: hole/valence (t_h) couples
# MORE than electron/conduction (t_e); ab-initio ratio t_h:t_e = 0.064:0.029.
# Their sum = AFM-FM gap shift (2.1 - 2.0 = 0.1 eV). (Previously these two were
# swapped: Jpd_v=0.2, Jpd_c=0.6.)
Jpd_v = 0.069                   # valence/hole coupling  (t_h)
Jpd_c = 0.031                   # conduction/electron coupling (t_e)
θ = π/4                         # fixed angle for the first benchmark

# Positive attraction magnitudes (the BSE kernel is negative). Fitted with the
# full continuum pole so the bright exciton is at 1.360 eV (AFM) and red-shifts
# 15 meV toward FM. Ratio V_intra/V_inter ≈ 2.47 ≈ ab-initio V_IN/V^CT = 2.48.
V_intra = 1.708                 # electron and hole in the same layer: |11>, |22>
V_inter = 0.691                 # electron and hole in different layers: |12>, |21>

# Thermodynamic and broadening parameters.
T = 0.01
μchem = Δ_0 / 2                 # inside the insulating gap; valence full, conduction empty
ηe = 0.03
ηΩ = 2ηe

# Frequency and momentum grids. The window covers the bound exciton (~1.36 eV),
# a second bound pole (~1.96 eV), and the e-h continuum onset (~2.0-2.1 eV).
Ωmin, Ωmax, NΩ = 0.0, 12.0, 601
ks = collect(range(-π, stop=π - 2π/L, length=L))
Ωs = collect(range(Ωmin, Ωmax, length=NΩ));

In [ ]:
# -----------------------------
# Matrices and one-particle Hamiltonian
# -----------------------------
P_v = ComplexF64[1 0; 0 0]
P_c = ComplexF64[0 0; 0 1]

σ_0 = ComplexF64[1 0; 0 1]
σ_x = ComplexF64[0 1; 1 0]
σ_y = ComplexF64[0 -1im; 1im 0]
σ_z = ComplexF64[1 0; 0 -1]

ϵk(k, γ) = -2γ * cos(k)

function H_valence(k, θ)
    εv = ϵk(k, γ_v)
    Jvθ = Jpd_v * cos(θ/2)
    return -(εv + 2γ_v + Jvθ) * σ_0 + Jvθ * σ_x
end

function H_conduction(k, θ)
    εc = ϵk(k, γ_c)
    cθ = cos(θ/2)
    Jcθ = Jpd_c * cθ
    gap_shift = (Jpd_c + Jpd_v) * (1 - cθ)
    return (εc + 2γ_c + Δ_0 + gap_shift + Jcθ) * σ_0 - Jcθ * σ_x
end

function H0(k, θ)
    return kron(P_v, H_valence(k, θ)) + kron(P_c, H_conduction(k, θ))
end

function fermi(E; T=T, μ=μchem)
    x = (E - μ) / T
    x > 60 && return 0.0
    x < -60 && return 1.0
    return 1 / (exp(x) + 1)
end;

## Band structure diagnostic

The bubble is evaluated in the eigenbasis of the valence and conduction blocks. This first plot checks the energy reference: the valence edge is at zero, and the bare gap is minimal in the ferromagnetic configuration.

In [ ]:
function diagonalize_model(ks, θ)
    Ev = zeros(Float64, 2, length(ks))
    Ec = zeros(Float64, 2, length(ks))
    Uv = Vector{Matrix{ComplexF64}}(undef, length(ks))
    Uc = Vector{Matrix{ComplexF64}}(undef, length(ks))

    for (ik, k) in enumerate(ks)
        Fv = eigen(Hermitian(H_valence(k, θ)))
        Fc = eigen(Hermitian(H_conduction(k, θ)))
        Ev[:, ik] .= Fv.values
        Ec[:, ik] .= Fc.values
        Uv[ik] = Matrix{ComplexF64}(Fv.vectors)
        Uc[ik] = Matrix{ComplexF64}(Fc.vectors)
    end

    return (; Ev, Ec, Uv, Uc, θ)
end

function transition_energies(data)
    ΔE = zeros(Float64, 2, 2, size(data.Ev, 2))
    for ik in axes(data.Ev, 2), a in 1:2, b in 1:2
        ΔE[a, b, ik] = data.Ec[a, ik] - data.Ev[b, ik]
    end
    return ΔE
end

function continuum_edges(data)
    ΔE = transition_energies(data)
    return minimum(ΔE), maximum(ΔE)
end;

In [ ]:
data = diagonalize_model(ks, θ)

fig, ax = subplots(figsize=(7.0, 4.2))
for a in 1:2
    ax.plot(ks, data.Ev[a, :], color="#2f7d32", lw=1.5, label=(a == 1 ? "valence" : nothing))
    ax.plot(ks, data.Ec[a, :], color="#b83232", lw=1.5, label=(a == 1 ? "conduction" : nothing))
end
ax.axhline(μchem, color="gray", ls="--", lw=1.0, label="mu")
ax.set_xlabel("k")
ax.set_ylabel("energy")
ax.set_title("Polarized one-particle bands, theta/pi = " * @sprintf("%.3f", θ/π))
ax.grid(alpha=0.18)
ax.legend(frameon=false)
fig.tight_layout()

## Layer-pair exciton basis for projection benchmarks

The interband response can also be projected onto the four layer-pair channels

$$\Gamma_{11}=|1_c\rangle\langle 1_v|,\quad
\Gamma_{22}=|2_c\rangle\langle 2_v|,$$

$$\Gamma_{12}=|1_c\rangle\langle 2_v|,\quad
\Gamma_{21}=|2_c\rangle\langle 1_v|.$$

The first two are intralayer channels. The last two are charge-transfer channels.

These $\Gamma_\alpha$ are only a convenient channel basis. The clean way to write the bubble is to insert the spectral projectors selected by each full electron-hole transition channel,

$$
P^\lambda_c(k)=|u^\lambda_c(k)\rangle\langle u^\lambda_c(k)|,
\qquad
P^\lambda_v(k)=|u^\lambda_v(k)\rangle\langle u^\lambda_v(k)|.
$$

Here $\lambda\in\{++ ,--,+-,-+\}$ labels the full electron-hole transition. The conduction and valence projectors above are the ones selected by that transition, with

$$
\Delta E^\lambda(k)=E^\lambda_c(k)-E^\lambda_v(k).
$$

The corresponding transition projector is

$$
Q^\lambda(k)=P^\lambda_c(k)\otimes P^\lambda_v(k).
$$

With the projectors inserted, the layer-pair bubble has only $\alpha,\beta$ as external indices,

$$
\Pi^R_{\Phi,\alpha\beta}(\Omega)=\frac{1}{L}\sum_{k,\lambda}
\frac{f(E^\lambda_v(k))-f(E^\lambda_c(k))}{\Omega+i\eta-\Delta E^\lambda(k)}
\operatorname{Tr}\!\left[
P^\lambda_c(k)\Gamma_\alpha P^\lambda_v(k)\Gamma_\beta^\dagger
\right].
$$

Equivalently, defining

$$
M_\alpha{}^\lambda(k)=\langle u^\lambda_c(k)|\Gamma_\alpha|u^\lambda_v(k)\rangle,
$$

the trace factor is $M_\alpha{}^\lambda(k)[M_\beta{}^\lambda(k)]^*$.

The band labels are therefore contained inside the summed projectors, so they never appear as external indices of the response.

The attractive interaction is diagonal in this basis, which is why it remains useful for benchmarks and physical channel weights:

$$V_\Phi=-\mathrm{diag}(V_{\rm intra},V_{\rm intra},V_{\rm inter},V_{\rm inter}).$$

## Diagonal transition basis and projection

For the derivation it is useful to first work in the transition basis that diagonalizes the bare electron-hole bubble. The four channels are labeled by

$$
\lambda\in\{++ ,--,+-,-+\}.
$$

The corresponding transition projectors

$$
Q^\lambda(k)=P^\lambda_c(k)\otimes P^\lambda_v(k)
$$

are the states that diagonalize the bare electron-hole bubble. The associated transition energy is

$$
\Delta E^\lambda(k)=E^\lambda_c(k)-E^\lambda_v(k).
$$

The momentum remains an internal variable in the bubble, and in this basis

$$
(\Pi^R_{\rm diag})^{\lambda\lambda'}(\Omega)=\delta_{\lambda\lambda'}\frac{1}{L}\sum_k
\frac{f(E^\lambda_v(k))-f(E^\lambda_c(k))}{\Omega+i\eta-\Delta E^\lambda(k)}.
$$

The layer-pair fields $\Phi_\alpha$, with $\alpha=(11,22,12,21)$, are obtained from the transition basis through the vertices

$$
M_\alpha{}^\lambda(k)=\langle u^\lambda_c(k)|\Gamma_\alpha|u^\lambda_v(k)\rangle.
$$

For $\alpha=(ll')$ this is explicitly

$$
M_{(ll')}{}^\lambda(k)=\langle u^\lambda_c(k)|l\rangle\langle l'|u^\lambda_v(k)\rangle.
$$

In the present model the $k$-dependent part of both one-particle Hamiltonians is proportional to $\sigma_0$, while the layer mixing is proportional to $\sigma_x$. Therefore the eigenvectors are independent of $k$ and are just the $\sigma_x$ eigenvectors,

$$
U_c=U_v=U=\frac{1}{\sqrt2}
\begin{pmatrix}
1&1\\
1&-1
\end{pmatrix},
$$

up to signs/phases and ordering conventions. Thus $M_\alpha{}^\lambda(k)=M_\alpha{}^\lambda$ is $k$ independent in this notebook. In the ordering $\Phi=(11,22,12,21)$ and $\lambda=(++),(--),(+-),(-+)$,

$$
M=\frac12
\begin{pmatrix}
1&1&1&1\\
1&1&-1&-1\\
1&-1&-1&1\\
1&-1&1&-1
\end{pmatrix}.
$$

Therefore the projected bubble is

$$
\Pi^R_\Phi(\Omega)=M\Pi^R_{\rm diag}(\Omega)M^\dagger.
$$

Working entirely in the diagonal transition basis is conceptually clean, but the interaction becomes

$$
V_{\rm diag}=M^\dagger V_\Phi M.
$$

Since the interaction is simple in the layer-pair basis,

$$V_\Phi=-\operatorname{diag}(V_{\rm intra},V_{\rm intra},V_{\rm inter},V_{\rm inter}),$$

the transformed kernel has a transparent block structure in the channel basis:

$$
V_{\rm diag}=\begin{pmatrix}
-\bar V & -\delta V & 0 & 0 \\
-\delta V & -\bar V & 0 & 0 \\
0 & 0 & -\bar V & -\delta V \\
0 & 0 & -\delta V & -\bar V
\end{pmatrix},
$$

with

$$
\bar V=\frac{V_{\rm intra}+V_{\rm inter}}{2},\qquad
\delta V=\frac{V_{\rm intra}-V_{\rm inter}}{2}.
$$

A different phase convention for the single-particle eigenvectors can flip the signs of the off-diagonal entries, but not the block structure or the eigenvalues.


In [ ]:
# -----------------------------
# Layer-pair vertices, transition-basis rotation, and interaction kernels
# -----------------------------
Γ_11 = ComplexF64[1 0; 0 0]
Γ_22 = ComplexF64[0 0; 0 1]
Γ_12 = ComplexF64[0 1; 0 0]
Γ_21 = ComplexF64[0 0; 1 0]

Γ_layer_pairs = [Γ_11, Γ_22, Γ_12, Γ_21]
layer_pair_labels = ["11", "22", "12", "21"]
transition_labels = ["++", "--", "+-", "-+"]

# The one-particle eigenvectors are the σ_x eigenvectors in this model.
U_layer = ComplexF64[1 1; 1 -1] ./ sqrt(2)

# Rows: Φ=(11,22,12,21). Columns: λ=(++),(--),(+-),(-+).
M_transition_to_layer = ComplexF64[
     1   1   1   1;
     1   1  -1  -1;
     1  -1  -1   1;
     1  -1   1  -1
] ./ 2

function interaction_kernel_layer_pair(; V_intra=V_intra, V_inter=V_inter)
    @assert V_intra != 0 "V_intra must be nonzero to invert V"
    @assert V_inter != 0 "V_inter must be nonzero to invert V"
    return -Diagonal(ComplexF64[V_intra, V_intra, V_inter, V_inter])
end

V_layer = interaction_kernel_layer_pair()
V_diag = M_transition_to_layer' * Matrix(V_layer) * M_transition_to_layer

function project_transition_to_layer(A_diag)
    A_layer = similar(A_diag)
    for iΩ in axes(A_diag, 3)
        A_layer[:, :, iΩ] .= M_transition_to_layer * A_diag[:, :, iΩ] * M_transition_to_layer'
    end
    return A_layer
end;


## Bare electron-hole bubble

For each momentum we diagonalize the valence and conduction blocks. The transition label $\lambda$ then selects one conduction projector and one valence projector. The retarded bare bubble is

$$
(\Pi^R_{\rm diag})^{\lambda\lambda'}(\Omega)=\delta_{\lambda\lambda'}\frac{1}{L}\sum_k
\frac{f(E^\lambda_v(k))-f(E^\lambda_c(k))}{\Omega+i\eta-\Delta E^\lambda(k)}.
$$

This is the Lehmann representation after the internal Langreth frequency integral has been performed analytically. The poles are the free electron-hole transition energies

$$
\Delta E^\lambda(k)=E^\lambda_c(k)-E^\lambda_v(k).
$$

The physical layer-pair bubble is the projection

$$
\Pi^R_{\Phi,\alpha\beta}(\Omega)
=\sum_{\lambda\lambda'}M_\alpha{}^\lambda(\Pi^R_{\rm diag})^{\lambda\lambda'}(\Omega)[M_\beta{}^{\lambda'}]^*.
$$

Since $\Pi^R_{\rm diag}$ is diagonal, this is equivalently

$$
\Pi^R_{\Phi,\alpha\beta}(\Omega;\theta)=\frac{1}{L}\sum_{k,\lambda}
\frac{f(E^\lambda_v(k))-f(E^\lambda_c(k))}{\Omega+i\eta-\Delta E^\lambda(k)}
M_\alpha{}^\lambda[M_\beta{}^\lambda]^*.
$$

Thus the bubble is diagonal before projection, and the off-diagonal channel structure in the $\Phi$ basis comes entirely from the matrix $M$.


In [ ]:
# -----------------------------
# Bare bubbles and dressed interaction
# -----------------------------
function transition_edges_diagonal(θval)
    E0 = Δ_0 + Jpd_c + Jpd_v
    cθ = cos(θval / 2)
    Jp = (Jpd_c + Jpd_v) * cθ
    Jm = (Jpd_c - Jpd_v) * cθ
    return Float64[E0 - Jp, E0 + Jp, E0 - Jm, E0 + Jm]
end

function diagonal_transition_energies(ks, θval)
    edges = transition_edges_diagonal(θval)
    A = 2 * (γ_c + γ_v)
    ΔE = zeros(Float64, length(edges), length(ks))
    for (ik, k) in enumerate(ks), λ in eachindex(edges)
        ΔE[λ, ik] = edges[λ] + A * (1 - cos(k))
    end
    return ΔE
end

function continuum_edges_diagonal(θval; ks=ks)
    ΔE = diagonal_transition_energies(ks, θval)
    return minimum(ΔE), maximum(ΔE)
end

function polarization_bare_diagonal(Ωs, ks, θval; η=ηΩ)
    ΔE = diagonal_transition_energies(ks, θval)
    nch = size(ΔE, 1)
    Π = zeros(ComplexF64, nch, nch, length(Ωs))

    # In the insulating equilibrium setup, f_v - f_c = 1 for all interband transitions.
    for λ in 1:nch, (ik, _) in enumerate(ks)
        Δ = ΔE[λ, ik]
        for (iΩ, Ω) in enumerate(Ωs)
            Π[λ, λ, iΩ] += 1 / (Ω + 1im * η - Δ) / length(ks)
        end
    end
    return Π, ΔE
end

function polarization_bare_layer_pair(Ωs, ks, θ; η=ηΩ, Γs=Γ_layer_pairs)
    data = diagonalize_model(ks, θ)
    nch = length(Γs)
    Π = zeros(ComplexF64, nch, nch, length(Ωs))

    for (ik, _) in enumerate(ks)
        Uv = data.Uv[ik]
        Uc = data.Uc[ik]
        Γe = [Uc' * Γ * Uv for Γ in Γs]

        for a in 1:2, b in 1:2
            ΔEab = data.Ec[a, ik] - data.Ev[b, ik]
            occ = fermi(data.Ev[b, ik]) - fermi(data.Ec[a, ik])

            for (iΩ, Ω) in enumerate(Ωs)
                pref = occ / (Ω + 1im * η - ΔEab) / length(ks)
                for α in 1:nch, β in 1:nch
                    Π[α, β, iΩ] += pref * Γe[α][a, b] * conj(Γe[β][a, b])
                end
            end
        end
    end

    return Π, data
end

function dressed_interaction(Π, Vmat)
    @assert size(Π, 1) == size(Vmat, 1)
    @assert size(Π, 2) == size(Vmat, 2)
    D = similar(Π)
    Vinv = inv(Matrix{ComplexF64}(Vmat))
    for iΩ in axes(Π, 3)
        D[:, :, iΩ] .= inv(Vinv - Π[:, :, iΩ])
    end
    return D
end

function spectral_trace(A)
    return [-2 * imag(tr(A[:, :, iΩ])) for iΩ in axes(A, 3)]
end

function component_spectrum(A, α)
    return [-2 * imag(A[α, α, iΩ]) for iΩ in axes(A, 3)]
end;


In [ ]:
println("Computing bare diagonal-transition bubble...")
@time Π_diag, ΔE_diag = polarization_bare_diagonal(Ωs, ks, θ)
D_diag = dressed_interaction(Π_diag, V_diag)

# Projection benchmark: same response represented in the original layer-pair basis.
Π_layer = project_transition_to_layer(Π_diag)
D_layer = project_transition_to_layer(D_diag)

# Optional direct layer-pair construction, used only as a consistency check.
Π_layer_direct, bubble_data = polarization_bare_layer_pair(Ωs, ks, θ)
projection_error = maximum(abs.(Π_layer .- Π_layer_direct))

cmin, cmax = continuum_edges_diagonal(θ; ks=ks)
println("theta/pi = ", θ/π)
println("V_diag = ")
show(stdout, "text/plain", V_diag); println()
println("bare electron-hole continuum: [", cmin, ", ", cmax, "]")
println("Pi_diag size: ", size(Π_diag))
println("D_diag size: ", size(D_diag))
println("finite Pi_diag: ", all(isfinite, Π_diag))
println("finite D_diag: ", all(isfinite, D_diag))
println("max |M Pi_diag M' - Pi_layer_direct| = ", projection_error)

println("max_Omega |Pi_{lambda lambda'}| in diagonal transition basis")
print("      ")
for label in transition_labels
    @printf("%12s", label)
end
println()
for λ in eachindex(transition_labels)
    @printf("%4s  ", transition_labels[λ])
    for λp in eachindex(transition_labels)
        @printf("%12.5e", maximum(abs.(Π_diag[λ, λp, :])))
    end
    println()
end


## Analytic k-integrated bare bubble

For the present model the transition basis is independent of momentum and each diagonal channel has the same one-dimensional cosine continuum,

$$
\Delta E^\lambda(k)=\Delta E^\lambda_0 + A(1-\cos k),\qquad
A=2(\gamma_c+\gamma_v).
$$

In the insulating equilibrium setup used here, the valence branch is full and the conduction branch is empty, so

$$
f(E^\lambda_v(k))-f(E^\lambda_c(k))=1.
$$

Therefore the momentum sum in the thermodynamic limit becomes

$$
(\Pi^R_{\rm diag})^{\lambda\lambda'}(\Omega)
=\delta_{\lambda\lambda'}\int_{-\pi}^{\pi}\frac{dk}{2\pi}
\frac{1}{\Omega+i\eta-\Delta E^\lambda_0-A(1-\cos k)}.
$$

Defining

$$
z_\lambda(\Omega)=\Omega+i\eta-\Delta E^\lambda_0-A,
$$

the integral is

$$
(\Pi^R_{\rm diag})^{\lambda\lambda}(\Omega)
=\int_{-\pi}^{\pi}\frac{dk}{2\pi}\frac{1}{z_\lambda+A\cos k}
=\frac{1}{\sqrt{z_\lambda^2-A^2}}.
$$

Equivalently,

$$
(\Pi^R_{\rm diag})^{\lambda\lambda}(\Omega)
=\frac{1}{\sqrt{(\Omega+i\eta-\Delta E^\lambda_0)
(\Omega+i\eta-\Delta E^\lambda_0-2A)}}.
$$

The square-root branch is the retarded one, chosen so that

$$
\sqrt{z_\lambda^2-A^2}\sim z_\lambda
$$

for large $|z_\lambda|$. This analytic expression includes the full continuum. It can replace the numerical $k$ sum in the pole equation $\det[V_{\rm diag}^{-1}-\Pi^R_{\rm diag}(\Omega)]=0$, but it is still frequency dependent and therefore is not the same as the finite-dimensional local BSE Hamiltonian. The local Hamiltonian appears only after replacing the continuum by the band-edge pole.


In [ ]:
function continuum_integral_1d_retarded(z, A)
    root = sqrt(z^2 - A^2)
    # Retarded branch: root behaves as z for large |z|. This fixes the sign
    # below the lower continuum edge, where the principal square root is positive.
    if real(z) < 0
        root = -root
    end
    return 1 / root
end

function polarization_bare_diagonal_analytic(Ωs, θval; η=ηΩ)
    edges = transition_edges_diagonal(θval)
    A = 2 * (γ_c + γ_v)
    Π = zeros(ComplexF64, length(edges), length(edges), length(Ωs))
    for λ in eachindex(edges), (iΩ, Ω) in enumerate(Ωs)
        zλ = Ω + 1im * η - edges[λ] - A
        Π[λ, λ, iΩ] = continuum_integral_1d_retarded(zλ, A)
    end
    return Π, edges
end

Π_diag_analytic, ΔE0_analytic = polarization_bare_diagonal_analytic(Ωs, θ)
D_diag_analytic = dressed_interaction(Π_diag_analytic, V_diag)
analytic_bubble_error = maximum(abs.(Π_diag .- Π_diag_analytic))
println("max |Pi k-sum - Pi analytic| = ", analytic_bubble_error)

fig, axs = subplots(1, 2, figsize=(12.0, 4.0))
for λ in eachindex(transition_labels)
    axs[1].plot(Ωs, real.(Π_diag[λ, λ, :]), lw=1.1, color=branch_colors[λ], label=transition_labels[λ] * " k sum")
    axs[1].plot(Ωs, real.(Π_diag_analytic[λ, λ, :]), lw=1.0, ls="--", color=branch_colors[λ], label=transition_labels[λ] * " analytic")

    axs[2].plot(Ωs, component_spectrum(Π_diag, λ), lw=1.1, color=branch_colors[λ], label=transition_labels[λ] * " k sum")
    axs[2].plot(Ωs, component_spectrum(Π_diag_analytic, λ), lw=1.0, ls="--", color=branch_colors[λ], label=transition_labels[λ] * " analytic")
end
axs[1].axvspan(cmin, cmax, color="gray", alpha=0.10)
axs[2].axvspan(cmin, cmax, color="gray", alpha=0.10)
axs[1].set_xlabel("Omega")
axs[2].set_xlabel("Omega")
axs[1].set_ylabel("Re Pi^R")
axs[2].set_ylabel("-2 Im Pi^R")
axs[1].set_title("Real part")
axs[2].set_title("Spectral weight")
for ax in axs
    ax.grid(alpha=0.18)
end
axs[2].legend(frameon=false, fontsize=6, ncol=2)
fig.suptitle("Numerical k sum vs analytic k-integrated bubble")
fig.tight_layout(rect=(0, 0, 1, 0.90))


## Figure 1: layer-pair BSE benchmark

This is the only figure kept in the original layer-pair representation. It is a benchmark showing that the diagonal-transition calculation can be projected back to the physical layer-pair channels,

$$D^R_\Phi=M D^R_{\rm diag}M^\dagger.$$

The rest of the notebook uses the diagonal transition basis as the primary computational basis.


In [ ]:
ΔE = ΔE_diag
AΠ = spectral_trace(Π_layer)
AD = spectral_trace(D_layer)

fig, axs = subplots(1, 3, figsize=(15.0, 4.2))
for λ in eachindex(transition_labels)
    axs[1].plot(ks, ΔE[λ, :], lw=1.35, color=branch_colors[λ], label=transition_labels[λ])
end
axs[1].axhline(cmin, color="gray", lw=0.9, ls="--")
axs[1].axhline(cmax, color="gray", lw=0.9, ls=":")
axs[1].set_xlabel("k")
axs[1].set_ylabel("E_c(k) - E_v(k)")
axs[1].set_title("Diagonal transition continua")
axs[1].grid(alpha=0.18)
axs[1].legend(frameon=false, fontsize=8)

for α in eachindex(layer_pair_labels)
    axs[2].plot(Ωs, component_spectrum(Π_layer, α), lw=1.4, label="Pi_" * layer_pair_labels[α])
end
axs[2].plot(Ωs, AΠ, color="black", lw=1.7, ls="--", label="Tr Pi")
axs[2].axvspan(cmin, cmax, color="gray", alpha=0.12)
axs[2].set_xlabel("Omega")
axs[2].set_ylabel("-2 Im Pi^R")
axs[2].set_title("Projected layer-pair bubble")
axs[2].grid(alpha=0.18)
axs[2].legend(frameon=false, fontsize=8)

for α in eachindex(layer_pair_labels)
    axs[3].plot(Ωs, component_spectrum(D_layer, α), lw=1.4, label="D_" * layer_pair_labels[α])
end
axs[3].plot(Ωs, AD, color="black", lw=1.7, ls="--", label="Tr D")
axs[3].axvspan(cmin, cmax, color="gray", alpha=0.12)
axs[3].set_xlabel("Omega")
axs[3].set_ylabel("-2 Im D^R")
axs[3].set_title("Projected layer-pair propagator")
axs[3].grid(alpha=0.18)
axs[3].legend(frameon=false, fontsize=8)

fig.suptitle("Layer-pair BSE benchmark from diagonal basis, theta/pi = " * @sprintf("%.3f", θ/π))
fig.tight_layout(rect=(0, 0, 1, 0.92))


## Figure 1b: diagonal-transition BSE benchmark

This is the same benchmark plotted directly in the diagonal transition basis. The first panel is the same set of free electron-hole continua. The second and third panels show the diagonal components of

$$
\Pi^R_{\rm diag}(\Omega),\qquad
D^R_{\rm diag}(\Omega)=\left[V_{\rm diag}^{-1}-\Pi^R_{\rm diag}(\Omega)\right]^{-1},
$$

for the channels $(++),(--),(+-),(-+)$.


In [ ]:
AΠ_diag = spectral_trace(Π_diag)
AD_diag = spectral_trace(D_diag)

fig, axs = subplots(1, 3, figsize=(15.0, 4.2))
for λ in eachindex(transition_labels)
    axs[1].plot(ks, ΔE_diag[λ, :], lw=1.35, color=branch_colors[λ], label=transition_labels[λ])
end
axs[1].axhline(cmin, color="gray", lw=0.9, ls="--")
axs[1].axhline(cmax, color="gray", lw=0.9, ls=":")
axs[1].set_xlabel("k")
axs[1].set_ylabel("E_c(k) - E_v(k)")
axs[1].set_title("Diagonal transition continua")
axs[1].grid(alpha=0.18)
axs[1].legend(frameon=false, fontsize=8)

for λ in eachindex(transition_labels)
    axs[2].plot(Ωs, component_spectrum(Π_diag, λ), lw=1.4, label="Pi_" * transition_labels[λ])
end
axs[2].plot(Ωs, AΠ_diag, color="black", lw=1.7, ls="--", label="Tr Pi")
axs[2].axvspan(cmin, cmax, color="gray", alpha=0.12)
axs[2].set_xlabel("Omega")
axs[2].set_ylabel("-2 Im Pi^R")
axs[2].set_title("Diagonal bare bubble")
axs[2].grid(alpha=0.18)
axs[2].legend(frameon=false, fontsize=8)

for λ in eachindex(transition_labels)
    axs[3].plot(Ωs, component_spectrum(D_diag, λ), lw=1.4, label="D_" * transition_labels[λ])
end
axs[3].plot(Ωs, AD_diag, color="black", lw=1.7, ls="--", label="Tr D")
axs[3].axvspan(cmin, cmax, color="gray", alpha=0.12)
axs[3].set_xlabel("Omega")
axs[3].set_ylabel("-2 Im D^R")
axs[3].set_title("Diagonal dressed propagator")
axs[3].grid(alpha=0.18)
axs[3].legend(frameon=false, fontsize=8)

fig.suptitle("Diagonal-transition BSE benchmark, theta/pi = " * @sprintf("%.3f", θ/π))
fig.tight_layout(rect=(0, 0, 1, 0.92))


## Full BSE and exciton poles

The starting point is the full Bethe-Salpeter/RPA equation for the excitonic susceptibility,

$$
\chi^R(\Omega)=\Pi^R(\Omega)+\Pi^R(\Omega)V\chi^R(\Omega),
$$

or equivalently

$$
\chi^R(\Omega)=\left[1-\Pi^R(\Omega)V\right]^{-1}\Pi^R(\Omega).
$$

These equations are basis independent. In the cells below, $\Pi^R$ and $V$ mean the diagonal-basis matrices unless a $\Phi$ subscript is written explicitly.

The dressed interaction has the same pole condition,

$$
D^R(\Omega)=\left[V^{-1}-\Pi^R(\Omega)\right]^{-1}.
$$

Therefore, before introducing any finite-dimensional Hamiltonian, an exciton is defined as a pole of the retarded response,

$$
\det\left[V^{-1}-\Pi^R(\Omega)\right]=0,
$$

or equivalently

$$
\det\left[1-\Pi^R(\Omega)V\right]=0.
$$

In the diagonal transition basis used below, the full momentum continuum is contained directly in $\Pi^R_{\rm diag}(\Omega)$ through the transition energies $\Delta E^\lambda(k)$.


## From \(\Pi^R\) to the BSE Hamiltonian

In the diagonal transition basis the full bubble keeps the momentum continuum,

$$
(\Pi^R_{\rm diag})^{\lambda\lambda}(\Omega)
=\frac{1}{L}\sum_k
\frac{1}{\Omega+i\eta-\Delta E^\lambda(k)}.
$$

A finite-dimensional BSE Hamiltonian appears only after the **band-edge approximation**. Since the basis does not depend on $k$ in this model, this approximation only replaces the dispersive transition energies by their lower-edge values,

$$
\Delta E^\lambda(k)\longrightarrow \Delta E^\lambda_0,
$$

so that

$$
\Pi^R_{\rm diag}(\Omega)
\longrightarrow
\Pi^R_{{\rm diag},0}(\Omega)
=\left[\Omega+i\eta-\Delta E_0(\theta)\right]^{-1}.
$$

Here $\Delta E_0$ is diagonal in the transition basis $\lambda=(++),(--),(+-),(-+)$:

$$
\Delta E_0(\theta)=\operatorname{diag}
\left(E_0-J_+,\ E_0+J_+,\ E_0-J_-,\ E_0+J_-\right),
$$

with

$$
E_0=\Delta_0+J_{pd,c}+J_{pd,v},\qquad
J_+=(J_{pd,c}+J_{pd,v})\cos(\theta/2),\qquad
J_-=(J_{pd,c}-J_{pd,v})\cos(\theta/2).
$$

The interaction in this basis is

$$
V_{\rm diag}=M^\dagger V_\Phi M,
\qquad
V_\Phi=-\operatorname{diag}(V_{\rm intra},V_{\rm intra},V_{\rm inter},V_{\rm inter}),
$$

or explicitly

$$
V_{\rm diag}=\begin{pmatrix}
-\bar V & -\delta V & 0 & 0 \\
-\delta V & -\bar V & 0 & 0 \\
0 & 0 & -\bar V & -\delta V \\
0 & 0 & -\delta V & -\bar V
\end{pmatrix},
$$

with

$$
\bar V=\frac{V_{\rm intra}+V_{\rm inter}}{2},\qquad
\delta V=\frac{V_{\rm intra}-V_{\rm inter}}{2}.
$$

Inserting $\Pi^R_{{\rm diag},0}$ into the pole condition gives

$$
\det\left[V_{\rm diag}^{-1}-(\Omega+i\eta-\Delta E_0)^{-1}\right]=0.
$$

Using determinant identities,

$$
\det\left[V_{\rm diag}^{-1}-(\Omega+i\eta-\Delta E_0)^{-1}\right]
\propto
\det\left[\Omega+i\eta-\Delta E_0-V_{\rm diag}\right].
$$

Therefore the local Hamiltonian is

$$
H_{\rm BSE}^{\rm loc}(\theta)=\Delta E_0(\theta)+V_{\rm diag}.
$$

Explicitly, in the ordering $(++),(--),(+-),(-+)$,

$$
H_{\rm BSE}^{\rm loc}(\theta)=
\begin{pmatrix}
E_0-J_+-\bar V & -\delta V & 0 & 0 \\
-\delta V & E_0+J_+-\bar V & 0 & 0 \\
0 & 0 & E_0-J_- -\bar V & -\delta V \\
0 & 0 & -\delta V & E_0+J_- -\bar V
\end{pmatrix}.
$$

The two $2\times2$ blocks are already visible in this basis. Their eigenvalues are

$$E_{s,\pm}^{\rm loc}=E_0-\bar V\pm\sqrt{\delta V^2+J_+^2},$$

$$E_{a,\pm}^{\rm loc}=E_0-\bar V\pm\sqrt{\delta V^2+J_-^2}.$$

So the conceptual chain is

$$
\Pi^R_{\rm diag}(\Omega)\ \text{full continuum bubble}
\rightarrow
\Pi^R_{{\rm diag},0}(\Omega)=(\Omega+i\eta-\Delta E_0)^{-1}
\rightarrow
H_{{\rm BSE},\,\rm diag}^{\rm loc}=\Delta E_0+V_{\rm diag}.
$$

The next cell compares the full $k$-summed bubble with this band-edge approximation.


## Band-edge approximation check

The full diagonal bubble keeps the continuum through the momentum sum,

$$
(\Pi^R_{\rm diag})^{\lambda\lambda}(\Omega)=\frac{1}{L}\sum_k
\frac{1}{\Omega+i\eta-\Delta E^\lambda(k)}.
$$

The local Hamiltonian approximation replaces each dispersive transition by its band-edge value,

$$
\Delta E^\lambda(k)\longrightarrow \Delta E^\lambda_0,
\qquad
(\Pi^R_{\rm diag})^{\lambda\lambda}(\Omega)
\longrightarrow
(\Pi^R_{{\rm diag},0})^{\lambda\lambda}(\Omega)=
\frac{1}{\Omega+i\eta-\Delta E^\lambda_0}.
$$

This is not an identity; it is the no-continuum reduction that turns the pole equation into the finite-dimensional local BSE Hamiltonian. The figure below compares the full $k$-summed object with this band-edge approximation at the benchmark angle.


In [ ]:
function polarization_bandedge_diagonal(Ωs, θval; η=ηΩ)
    edges = transition_edges_diagonal(θval)
    Π0 = zeros(ComplexF64, length(edges), length(edges), length(Ωs))
    for λ in eachindex(edges), (iΩ, Ω) in enumerate(Ωs)
        Π0[λ, λ, iΩ] = 1 / (Ω + 1im * η - edges[λ])
    end
    return Π0, edges
end

Π_diag0, ΔE0_diag = polarization_bandedge_diagonal(Ωs, θ)
D_diag0 = dressed_interaction(Π_diag0, V_diag)

fig, axs = subplots(1, 3, figsize=(15.0, 4.2))
for λ in eachindex(transition_labels)
    axs[1].plot(ks, ΔE_diag[λ, :], lw=1.25, color=branch_colors[λ], label=transition_labels[λ] * " full")
    axs[1].axhline(ΔE0_diag[λ], lw=1.05, ls="--", color=branch_colors[λ])
end
axs[1].set_xlabel("k")
axs[1].set_ylabel("transition energy")
axs[1].set_title("Full continuum vs band edge")
axs[1].grid(alpha=0.18)
axs[1].legend(frameon=false, fontsize=7)

for λ in eachindex(transition_labels)
    axs[2].plot(Ωs, component_spectrum(Π_diag, λ), lw=1.3, color=branch_colors[λ], label=transition_labels[λ] * " full")
    axs[2].plot(Ωs, component_spectrum(Π_diag0, λ), lw=1.0, ls="--", color=branch_colors[λ], label=transition_labels[λ] * " edge")
end
axs[2].axvspan(cmin, cmax, color="gray", alpha=0.10)
axs[2].set_xlabel("Omega")
axs[2].set_ylabel("-2 Im Pi^R")
axs[2].set_title("Bare bubble")
axs[2].grid(alpha=0.18)
axs[2].legend(frameon=false, fontsize=6, ncol=2)

for λ in eachindex(transition_labels)
    axs[3].plot(Ωs, component_spectrum(D_diag, λ), lw=1.3, color=branch_colors[λ], label=transition_labels[λ] * " full")
    axs[3].plot(Ωs, component_spectrum(D_diag0, λ), lw=1.0, ls="--", color=branch_colors[λ], label=transition_labels[λ] * " edge")
end
axs[3].axvspan(cmin, cmax, color="gray", alpha=0.10)
axs[3].set_xlabel("Omega")
axs[3].set_ylabel("-2 Im D^R")
axs[3].set_title("Dressed propagator")
axs[3].grid(alpha=0.18)
axs[3].legend(frameon=false, fontsize=6, ncol=2)

fig.suptitle("Full k-summed diagonal BSE vs band-edge approximation, theta/pi = " * @sprintf("%.3f", θ/π))
fig.tight_layout(rect=(0, 0, 1, 0.92))


In [ ]:
# -----------------------------
# Full-propagator pole benchmarks and local BSE Hamiltonian limit
# -----------------------------
E_gap_analytic(θval) = Δ_0 + (Jpd_c + Jpd_v) * (1 - cos(θval/2))
E0_pair_analytic() = Δ_0 + Jpd_c + Jpd_v
J_plus_analytic(θval) = (Jpd_c + Jpd_v) * cos(θval/2)
J_minus_analytic(θval) = (Jpd_c - Jpd_v) * cos(θval/2)
pair_dispersion_scale() = 2 * (γ_c + γ_v)

function continuum_g_below(Ω, Eedge; γc=γ_c, γv=γ_v)
    A = 2 * (γc + γv)
    Ω < Eedge || return NaN
    radicand = (Eedge + A - Ω)^2 - A^2
    radicand > 0 || return NaN
    return -1 / sqrt(radicand)
end

function sector_pole_equation(Ω, Jsector; V_intra=V_intra, V_inter=V_inter)
    E0 = E0_pair_analytic()
    Jabs = abs(Jsector)
    Vavg = 0.5 * (V_intra + V_inter)
    gminus = continuum_g_below(Ω, E0 - Jabs)
    gplus = continuum_g_below(Ω, E0 + Jabs)
    if !(isfinite(gminus) && isfinite(gplus))
        return NaN
    end
    return 1 + Vavg * (gminus + gplus) + V_intra * V_inter * gminus * gplus
end

function bisect_root(f, a, b; iterations=80)
    fa = f(a)
    fb = f(b)
    @assert isfinite(fa) && isfinite(fb)
    @assert fa * fb <= 0
    lo, hi = a, b
    flo = fa
    for _ in 1:iterations
        mid = 0.5 * (lo + hi)
        fmid = f(mid)
        if !isfinite(fmid)
            hi = mid
        elseif flo * fmid <= 0
            hi = mid
            fb = fmid
        else
            lo = mid
            flo = fmid
        end
    end
    return 0.5 * (lo + hi)
end

function sector_bound_roots(Jsector; V_intra=V_intra, V_inter=V_inter,
                            nscan=2500, margin=1e-7, search_window=nothing)
    E0 = E0_pair_analytic()
    Jabs = abs(Jsector)
    lower_edge = E0 - Jabs
    A = pair_dispersion_scale()
    width = isnothing(search_window) ? 4A + 4max(abs(V_intra), abs(V_inter)) + 2Jabs + 2 : search_window
    xs = collect(range(lower_edge - width, stop=lower_edge - margin, length=nscan))
    f(Ω) = sector_pole_equation(Ω, Jsector; V_intra=V_intra, V_inter=V_inter)

    roots = Float64[]
    xprev = xs[1]
    fprev = f(xprev)
    for x in xs[2:end]
        fx = f(x)
        if isfinite(fprev) && isfinite(fx)
            if fprev == 0
                push!(roots, xprev)
            elseif fprev * fx < 0
                push!(roots, bisect_root(f, xprev, x))
            end
        end
        xprev = x
        fprev = fx
    end

    sort!(roots)
    unique_roots = Float64[]
    for r in roots
        if isempty(unique_roots) || abs(r - unique_roots[end]) > 1e-5
            push!(unique_roots, r)
        end
    end
    return unique_roots
end

function local_transition_energies(θval)
    return sort(transition_edges_diagonal(θval))
end

function local_bse_hamiltonian(θval; V_intra=V_intra, V_inter=V_inter)
    ΔE0 = Diagonal(ComplexF64.(transition_edges_diagonal(θval)))
    Vd = M_transition_to_layer' * Matrix(interaction_kernel_layer_pair(; V_intra=V_intra, V_inter=V_inter)) * M_transition_to_layer
    return Matrix(ΔE0) + Vd
end

function local_bse_sector_matrices(θval; V_intra=V_intra, V_inter=V_inter)
    E0 = E0_pair_analytic()
    Vavg = 0.5 * (V_intra + V_inter)
    δV = 0.5 * (V_intra - V_inter)
    Jp = J_plus_analytic(θval)
    Jm = J_minus_analytic(θval)
    Hs = ComplexF64[E0 - Jp - Vavg   -δV;
                    -δV              E0 + Jp - Vavg]
    Ha = ComplexF64[E0 - Jm - Vavg   -δV;
                    -δV              E0 + Jm - Vavg]
    return Hs, Ha
end

function local_bse_branches(θval; V_intra=V_intra, V_inter=V_inter)
    E0 = E0_pair_analytic()
    Vavg = 0.5 * (V_intra + V_inter)
    δV = 0.5 * (V_intra - V_inter)
    Jp = J_plus_analytic(θval)
    Jm = J_minus_analytic(θval)
    Esplit = sqrt(δV^2 + Jp^2)
    Asplit = sqrt(δV^2 + Jm^2)
    return (; Es_minus=E0 - Vavg - Esplit,
            Es_plus =E0 - Vavg + Esplit,
            Ea_minus=E0 - Vavg - Asplit,
            Ea_plus =E0 - Vavg + Asplit)
end

local_band_edge_branches(θval; kwargs...) = local_bse_branches(θval; kwargs...)
local_bse_eigenvalues(θval; kwargs...) = sort(real(eigvals(Hermitian(local_bse_hamiltonian(θval; kwargs...)))))

function analytic_branch_arrays(θs)
    local_branches = zeros(Float64, 4, length(θs))
    local_eigs = zeros(Float64, 4, length(θs))
    bound_poles = fill(NaN, 4, length(θs))
    sector_edges = zeros(Float64, 2, length(θs))
    Egap = zeros(Float64, length(θs))
    Jplus = zeros(Float64, length(θs))
    Jminus = zeros(Float64, length(θs))

    for (i, θval) in enumerate(θs)
        b = local_bse_branches(θval)
        local_branches[:, i] .= [b.Es_minus, b.Ea_minus, b.Ea_plus, b.Es_plus]
        local_eigs[:, i] .= local_bse_eigenvalues(θval)
        Egap[i] = E_gap_analytic(θval)
        Jplus[i] = J_plus_analytic(θval)
        Jminus[i] = J_minus_analytic(θval)
        sector_edges[:, i] .= [E0_pair_analytic() - abs(Jplus[i]),
                               E0_pair_analytic() - abs(Jminus[i])]

        sroots = sector_bound_roots(Jplus[i])
        aroots = sector_bound_roots(Jminus[i])
        for j in 1:min(2, length(sroots))
            bound_poles[j, i] = sroots[j]
        end
        for j in 1:min(2, length(aroots))
            bound_poles[2 + j, i] = aroots[j]
        end
    end

    local_labels = ["E_s- local", "E_a- local", "E_a+ local", "E_s+ local"]
    pole_labels = ["s pole 1", "s pole 2", "a pole 1", "a pole 2"]
    sector_labels = ["s lower edge", "a lower edge"]
    return (; branches=local_branches, local_branches, local_eigs,
            labels=local_labels, local_labels,
            bound_poles, pole_labels, sector_edges, sector_labels,
            Egap, Jplus, Jminus)
end;

In [ ]:
# -----------------------------
# Calibration check: full-continuum exciton poles vs CrSBr measurements
# Targets (arXiv:2409.18501): bright A exciton at 1.360 eV (AFM), 15 meV red-shift to FM.
# The bright exciton is the symmetric sector (J_+); the dark partner is the
# antisymmetric sector (J_-). Energies are roots of the full continuum equation
# Sector pole equations are solved by sector_bound_roots -- the SAME poles drawn in Figs 2 and 4
# (not the local-Hamiltonian eigenvalues, which under-bind by ~1 eV here).
# -----------------------------
function bright_dark_poles(θval)
    s = sector_bound_roots(J_plus_analytic(θval))
    a = sector_bound_roots(J_minus_analytic(θval))
    return (isempty(s) ? NaN : minimum(s)), (isempty(a) ? NaN : minimum(a))
end

Es_AFM, Ea_AFM = bright_dark_poles(π)
Es_FM,  Ea_FM  = bright_dark_poles(0.0)
println("Calibration check (full BSE/continuum poles):")
@printf("  bright (A) exciton:  AFM = %.4f eV   FM = %.4f eV   red-shift = %5.1f meV  [target 1.360 / 15]\n",
        Es_AFM, Es_FM, 1e3*(Es_AFM - Es_FM))
@printf("  dark partner:        AFM = %.4f eV   FM = %.4f eV   red-shift = %5.1f meV\n",
        Ea_AFM, Ea_FM, 1e3*(Ea_AFM - Ea_FM))
@printf("  V_intra/V_inter = %.3f   (ab-initio V_IN/V^CT ≈ 2.48)\n", V_intra/V_inter)
@printf("  AFM gap E0 = %.3f eV   FM gap Δ0 = %.3f eV   continuum scale A = %.2f eV\n",
        E0_pair_analytic(), Δ_0, pair_dispersion_scale())

## Angular sweep

We now sweep the relative magnetic angle using the diagonal transition basis. For each angle we store

$$A_D(\Omega,\theta)=-2\operatorname{Im}\operatorname{Tr}D^R_{\rm diag}(\Omega;\theta),$$

and the diagonal-transition resolved spectra

$$A^\lambda(\Omega,\theta)=-2\operatorname{Im}(D^R_{\rm diag})^{\lambda\lambda}(\Omega;\theta),
\qquad \lambda=(++),(--),(+-),(-+).$$

The layer-pair representation is no longer used for the main sweep; it was checked in the benchmark above.


In [ ]:
# -----------------------------
# Angular sweep in the diagonal transition basis
# -----------------------------
function solve_angle_diagonal(θval; Ωs=Ωs, ks=ks, Vmat=V_diag)
    Πθ, ΔEθ = polarization_bare_diagonal(Ωs, ks, θval)
    Dθ = dressed_interaction(Πθ, Vmat)
    ADθ = spectral_trace(Dθ)
    AΠθ = spectral_trace(Πθ)
    Acompθ = zeros(Float64, length(transition_labels), length(Ωs))
    for λ in 1:length(transition_labels)
        Acompθ[λ, :] .= component_spectrum(Dθ, λ)
    end
    detθ = [abs(det(inv(Matrix{ComplexF64}(Vmat)) - Πθ[:, :, iΩ])) for iΩ in axes(Πθ, 3)]
    cmin, cmax = minimum(ΔEθ), maximum(ΔEθ)
    ipeak = argmax(ADθ)
    idet = argmin(detθ)
    return (; θ=θval, Π=Πθ, D=Dθ, AD=ADθ, AΠ=AΠθ, Acomp=Acompθ,
            detvals=detθ, Ωpeak=Ωs[ipeak], Apeak=ADθ[ipeak],
            Ωdet=Ωs[idet], detmin=detθ[idet], continuum_min=cmin,
            continuum_max=cmax, ΔE=ΔEθ)
end

function sweep_angles_diagonal(θs; Ωs=Ωs, ks=ks, Vmat=V_diag)
    results = [solve_angle_diagonal(θval; Ωs=Ωs, ks=ks, Vmat=Vmat) for θval in θs]
    ADmap = hcat([r.AD for r in results]...)
    AΠmap = hcat([r.AΠ for r in results]...)
    Acomp = zeros(Float64, length(transition_labels), length(Ωs), length(θs))
    for (iθ, r) in enumerate(results)
        Acomp[:, :, iθ] .= r.Acomp
    end
    return (; θs, results, ADmap, AΠmap, Acomp,
            Ωpeak=[r.Ωpeak for r in results], Apeak=[r.Apeak for r in results],
            Ωdet=[r.Ωdet for r in results], detmin=[r.detmin for r in results],
            continuum_min=[r.continuum_min for r in results],
            continuum_max=[r.continuum_max for r in results])
end;


In [ ]:
# Editable angular sweep parameters.
Nθ = 41
θs_sweep = collect(range(0.0, stop=π, length=Nθ))
analytic_sweep = analytic_branch_arrays(θs_sweep)

println("Computing angular sweep in the diagonal transition basis...")
@time sweep = sweep_angles_diagonal(θs_sweep)
println("AD map size: ", size(sweep.ADmap))
println("component map size: ", size(sweep.Acomp))
println("finite AD map: ", all(isfinite, sweep.ADmap))
println("strongest spectral peak range: ", extrema(sweep.Ωpeak))
println("determinant-minimum range: ", extrema(sweep.Ωdet))
println("continuum lower edge range: ", extrema(sweep.continuum_min))
println("local BSE eigenvalue ranges:")
for ib in axes(analytic_sweep.local_branches, 1)
    println("  ", analytic_sweep.local_labels[ib], ": ", extrema(analytic_sweep.local_branches[ib, :]))
end
println("full-propagator pole ranges:")
for ip in axes(analytic_sweep.bound_poles, 1)
    vals = filter(isfinite, analytic_sweep.bound_poles[ip, :])
    if !isempty(vals)
        println("  ", analytic_sweep.pole_labels[ip], ": ", extrema(vals))
    end
end
println("max formula/eigenvalue mismatch: ", maximum(abs.(analytic_sweep.local_branches - analytic_sweep.local_eigs)))


## Figure 2: full BSE/dressed-propagator angular map

This is the primary object: the spectrum of the full retarded BSE propagator in the diagonal transition basis,

$$D^R_{\rm diag}(\Omega)=\left[V_{\rm diag}^{-1}-\Pi^R_{\rm diag}(\Omega)\right]^{-1}.$$

The dashed curves are the analytic bound-state roots of the continuum pole equation. In this figure the exciton is identified as a pole of the full response, before making the finite-dimensional Hamiltonian approximation.


In [ ]:
fig, ax = subplots(figsize=(7.4, 4.8))
extent = (minimum(θs_sweep) / π, maximum(θs_sweep) / π, minimum(Ωs), maximum(Ωs))
# NOTE: never name this variable `im` -- that shadows Julia's imaginary unit
# and breaks every later `1im` (e.g. inside the bubble) on the next cell run.
img = ax.imshow(sweep.ADmap, origin="lower", aspect="auto", extent=extent, cmap="magma")

# Full-continuum bound poles overlaid on the dressed spectrum. These are the
# physical excitons: "s pole 1" is the bright (A) exciton, "a pole 1" its dark
# partner; "pole 2" entries are the second bound states near the continuum edge.
for ip in axes(analytic_sweep.bound_poles, 1)
    y = analytic_sweep.bound_poles[ip, :]
    if any(isfinite.(y))
        ax.plot(θs_sweep ./ π, y, color=pole_colors[ip],
                lw=1.5, ls="--", label=analytic_sweep.pole_labels[ip])
    end
end

ax.set_xlabel("theta / pi")
ax.set_ylabel("Omega")
ax.set_title("Full BSE propagator: exciton poles")
ax.legend(frameon=false, fontsize=8, loc="upper right")
ax.set_ylim(1.2, 2.4)
cbar = fig.colorbar(img, ax=ax)
cbar.set_label("-2 Im Tr D^R")
fig.tight_layout()

## Figure 3: local BSE Hamiltonian limit

After replacing the diagonal transition continua by their band-edge values, the local bubble is

$$
\Pi^R_{{\rm diag},0}(\Omega)=\left[\Omega+i\eta-\Delta E_0(\theta)\right]^{-1}.
$$

The local Hamiltonian pole condition is therefore

$$
\det\left[\Omega+i\eta-(\Delta E_0+V_{\rm diag})\right]=0.
$$

The curves below are the eigenvalues of this approximate Hamiltonian representation.


In [ ]:
fig, axs = subplots(1, 2, figsize=(10.8, 3.9))
# branch_colors / pole_colors are defined once in the setup cell.
for ib in axes(analytic_sweep.local_branches, 1)
    axs[1].plot(θs_sweep ./ π, analytic_sweep.local_branches[ib, :],
                color=branch_colors[ib], lw=1.9, label=analytic_sweep.local_labels[ib])
end
#axs[1].plot(θs_sweep ./ π, analytic_sweep.Egap, color="gray", lw=1.2, ls=":", label="bare gap")
axs[1].set_xlabel("theta / pi")
axs[1].set_ylabel("energy")
axs[1].set_title("Local BSE Hamiltonian eigenvalues")
axs[1].grid(alpha=0.18)
axs[1].legend(frameon=false, fontsize=8)
#axs[1].set_ylim(1.3,1.5)

axs[2].plot(θs_sweep ./ π, analytic_sweep.Jplus, color="#4c78a8", lw=1.8, label="J_plus")
axs[2].plot(θs_sweep ./ π, analytic_sweep.Jminus, color="#f58518", lw=1.8, label="J_minus")
axs[2].set_xlabel("theta / pi")
axs[2].set_ylabel("mixing")
#axs[1].set_ylim(1.7,4.0)

axs[2].set_title("Angle-dependent BSE mixing")
axs[2].grid(alpha=0.18)
axs[2].legend(frameon=false, fontsize=8)
fig.tight_layout()

## Figure 4: full poles versus Hamiltonian approximation

This diagnostic puts both descriptions on the same axis. Dashed curves are poles of the full BSE propagator with the $k$ continuum included. Solid curves are eigenvalues of the projected local Hamiltonian obtained after the additional approximations. Their mismatch is therefore physical within the approximation hierarchy, not a contradiction.

In [ ]:
fig, ax = subplots(figsize=(7.4, 4.4))
for ip in axes(analytic_sweep.bound_poles, 1)
    y = analytic_sweep.bound_poles[ip, :]
    if any(isfinite.(y))
        ax.plot(θs_sweep ./ π, y, color=pole_colors[ip], lw=1.5, ls="--",
                label=analytic_sweep.pole_labels[ip])
    end
end
for ib in axes(analytic_sweep.local_branches, 1)
    ax.plot(θs_sweep ./ π, analytic_sweep.local_branches[ib, :],
            color=branch_colors[ib], lw=1.3, alpha=0.85,
            label=analytic_sweep.local_labels[ib])
end
ax.plot(θs_sweep ./ π, sweep.Ωdet, color="black", lw=1.2, ls=":", label="det minimum")
ax.plot(θs_sweep ./ π, sweep.Ωpeak, color="gray", lw=1.2, ls="-.", label="strongest peak")
ax.set_xlabel("theta / pi")
ax.set_ylabel("energy")
ax.set_title("Full BSE poles and local Hamiltonian limit")
ax.set_ylim(0.0, 3.0)
ax.grid(alpha=0.18)
ax.legend(frameon=false, fontsize=7, ncol=2)
fig.tight_layout()

## Figure 5: diagonal-transition channel-resolved spectra

These four maps show how the spectral weight is distributed between the diagonal transition channels $(++),(--),(+-),(-+)$ as the magnetic angle changes. Layer-pair weights can still be obtained by applying $M$ afterward, but they are not the primary basis for the calculation.


In [ ]:
fig, axs = subplots(2, 2, figsize=(10.0, 7.0), sharex=true, sharey=true)
axs_flat = vec(axs)
max_abs = maximum(abs.(sweep.Acomp))
imgg = nothing                  # pre-declare so the loop (soft global scope) can fill it
for λ in eachindex(transition_labels)
    ax = axs_flat[λ]
    imgg = ax.imshow(sweep.Acomp[λ, :, :], origin="lower", aspect="auto", extent=extent,
                   cmap="magma", vmin=0.0, vmax=max_abs)
    ax.plot(θs_sweep ./ π, sweep.continuum_min, color="#d8d8d8", lw=1.0, ls=":")
    ax.set_title("D_" * transition_labels[λ] * transition_labels[λ])
    ax.grid(false)
end
for ax in axs[:, 1]
    ax.set_ylabel("Omega")
end
for ax in axs[end, :]
    ax.set_xlabel("theta / pi")
end
cbar = fig.colorbar(imgg, ax=axs_flat, shrink=0.86)
cbar.set_label("-2 Im D^R^{λλ}")
fig.suptitle("Diagonal-transition resolved dressed spectra")
fig.tight_layout(rect=(0, 0, 1, 0.94))
